# BLISS AstroPy Affiliate Package Tutorial

## Introduction

Bayesian Light Source Separator (BLISS) is a Bayesian method for deblending and cataloging light sources.

## Installation

In [ ]:
%env BLISS_HOME=/home/zhteoh/730-astropy-integration

In [ ]:
!pip install -e $BLISS_HOME

# Tutorial

## Train the model

### Generate synthetic image data

In [ ]:
from bliss.api import generate

In [ ]:
generate(
    n_batches=3, 
    batch_size=64, 
    max_images_per_file=128, 
    cached_data_path="/data/scratch/zhteoh/730-tutorial/dataset"
)

#### Pass additional custom configuration parameters

In [ ]:
generate(
    n_batches=3,  # required
    batch_size=64,  # required
    max_images_per_file=128,  # required
    cached_data_path="/data/scratch/zhteoh/730-tutorial/dataset",  # required
    simulator={"prior": {"mean_sources": 0.02}},  # optional
    generate={"file_prefix": "dataset"},  # optional
)

In [ ]:
# Check that the dataset is generated
!ls /data/scratch/zhteoh/730-tutorial/dataset
!du -sh /data/scratch/zhteoh/730-tutorial/dataset
# !cat /data/scratch/zhteoh/730-tutorial/dataset/hparams.yaml

import torch
with open("/data/scratch/zhteoh/730-tutorial/dataset/dataset_0.pt", "rb") as f:
    dataset = torch.load(f)
print(len(dataset))
print(dataset[0]["images"].shape)

### Train the model

In [ ]:
from bliss.api import train

#### Without pretrained weights

In [ ]:
train(
    weight_save_path="/data/scratch/zhteoh/730-tutorial/output/tutorial_encoder/0.pt",
)

#### With pretrained weights

Download our relevant pretrained weights for your sky survey.

In [ ]:
from bliss.api import load_pretrained_weights_for_survey

import os
assert os.path.exists("/data/scratch/zhteoh/730-tutorial/pretrained_weights")

load_pretrained_weights_for_survey(
    survey="sdss",
    pretrained_weights_path="/data/scratch/zhteoh/730-tutorial/pretrained_weights/sdss_pretrained.pt",
)

#### Train on cached generated disk dataset

In [ ]:
from bliss.api import train_on_cached_data

In [ ]:
train_on_cached_data(
    weight_save_path="/data/scratch/zhteoh/730-tutorial/output/tutorial_encoder/0.pt",
    cached_data_path="/data/scratch/zhteoh/730-tutorial/dataset",
    train_n_batches=2,
    batch_size=64,
    val_split_file_idxs=[1],
    training={"pretrained_weights": "/data/scratch/zhteoh/730-tutorial/pretrained_weights/sdss_pretrained.pt"}
)

## Run the model

### Using sample dataset

#### Download the sample dataset

In [ ]:
from bliss.surveys.sdss import PhotoFullCatalog, SloanDigitalSkySurvey

sdss_data_path = "/home/zhteoh/730-astropy-integration/data/sdss"
photo_cat = PhotoFullCatalog.from_file(sdss_data_path, run=94, camcol=1, field=12, band=2)
sdss = SloanDigitalSkySurvey(sdss_data_path, 94, 1, (12,), (2,))

#### Get predictions for the sample dataset

In [ ]:
from bliss.api import predict_sdss

est_cat, est_cat_table, galaxy_params_table = predict_sdss(
    data_path="/home/zhteoh/730-astropy-integration/case_studies/astropy_integration_730/data/sdss", 
    weight_save_path="/home/zhteoh/730-astropy-integration/case_studies/astropy_integration_730/data/pretrained_models/sdss.pt",
    # predict={"dataset": {"run": 94, "camcol": 1, "fields": [12]}}
)

In [ ]:
from IPython.display import display
from IPython.core.display import HTML

with open("./predict.html", "r") as f:
    html_str = f.read()
    display(HTML(html_str))

In [ ]:
est_cat_table.show_in_notebook(display_length=5)

In [ ]:
galaxy_params_table.show_in_notebook(display_length=5)

#### Save predicted catalog to FITS file

In [ ]:
est_cat_table.write("est_cat.fits", format="fits")

In [ ]:
# Check that catalog is saved as intended
from astropy.table import Table

est_cat_table = Table.read("est_cat.fits", format="fits")
est_cat_table.show_in_notebook(display_length=5)

#### Evaluate prediction

In [ ]:
import torch

from bliss.metrics import BlissMetrics

est_cat_cuda = est_cat.to(torch.device("cpu"))
photo_cat_cuda = photo_cat.to(torch.device("cpu"))

metrics = BlissMetrics()
results = metrics(est_cat_cuda, photo_cat_cuda)

print(results)

### Using user-specified dataset

#### Download online dataset

In [ ]:
from astropy.coordinates import SkyCoord
from astroquery.sdss import SDSS
from pathlib import Path

from bliss.api import load_survey

# pos = SkyCoord('0h8m05.63s +14d50m23.3s', frame='icrs') # 1011/3/44
# pos = SkyCoord("1h8m05.73s +13d10m20.3s", frame="icrs") # 4829/5/27
pos = SkyCoord("1h2m05.83s -2d11m20.3s", frame="icrs") # 2699/4/71
region = SDSS.query_region(pos, radius="5 arcsec")
run, camcol, field = region["run"][0], region["camcol"][0], region["field"][0]
print("run:", run, "camcol:", camcol, "field:", field)
load_survey("sdss", run, camcol, field, download_dir=Path("/home/zhteoh/730-astropy-integration/case_studies/astropy_integration_730/data/sdss"))

#### Get predictions for the downloaded dataset

In [ ]:
from bliss.api import predict_sdss

est_cat_dl, est_cat_table_dl, galaxy_params_table_dl = predict_sdss(
    data_path="/home/zhteoh/730-astropy-integration/case_studies/astropy_integration_730/data/sdss",
    weight_save_path="/home/zhteoh/730-astropy-integration/case_studies/astropy_integration_730/data/pretrained_models/sdss.pt",
    predict={"dataset": {"run": 1011, "camcol": 3, "fields": [44]}}
)

In [ ]:
est_cat_table_dl.show_in_notebook(display_length=5)

In [ ]:
galaxy_params_table_dl.show_in_notebook(display_length=5)